# Notebook 04: Structured Outputs & JSON Schema

**Objectives:**
- Extract structured data with json_extract.v1 prompt
- Validate against JSON schemas
- Repair malformed JSON
- Log token costs for validation + repair cycles

In [ ]:
import sys
import pprint
sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model
from utils.json_utils import safe_parse_json, validate_json_schema, create_simple_schema, format_schema_for_prompt
import json

## Part 1: JSON Extraction with Schema

Define a schema and extract structured data.

In [ ]:
text = """The CloudSync Pro Business plan costs 20 LKR per user per month.
It includes 10TB storage and is currently available."""

schema = create_simple_schema({
                            "name": "string",
                            "price": "number",
                            "currency": "string",
                            "in_stock": "boolean"
                        }, required=["name", "price", "currency"])
pprint.pprint(schema)

In [ ]:
prompt_text, spec = render("json_extract.v1", schema=schema, text=text)
print(prompt_text)

In [ ]:
model = pick_model('google', 'general')
client = LLMClient('google', model)

response = client.json_chat(
    [
        {"role": "user", "content": prompt_text}
    ],
    temperature=0.0
)['text']

Handle malformed JSON with automatic repair.

In [ ]:
success, data, error = safe_parse_json(response)
pprint.pprint(data)

## Part 2: Pydantic Models (Recommended Approach)

Use Pydantic for type-safe structured outputs with automatic validation and IDE support.


In [ ]:
from pydantic import BaseModel, Field
from utils.json_utils import (
                            format_pydantic_schema_for_prompt,
                            parse_json_with_pydantic
)

class ProductInfo(BaseModel):
    name: str = Field(..., description="The name of the product")
    price: float = Field(..., description="The price of the product")
    currency: str = Field(..., description="The currency of the product")
    in_stock: bool = Field(..., description="Whether the product is in stock")


schema_str = format_pydantic_schema_for_prompt(ProductInfo)
print(schema_str)

In [ ]:
from pandas.io.formats.printing import _pprint_seq


text = """The CloudSync Pro Business plan costs $20 per user per month.
It includes 10TB storage and is currently available."""

prompt_text, spec = render("json_extract.v1", schema=schema_str, text=text)

model = pick_model('google', 'general')
client = LLMClient('google', model)

response = client.json_chat(
    [
        {"role": "user", "content": prompt_text}
    ],
    temperature=0.0
)['text']
print(response)

In [ ]:
success, data, error = safe_parse_json(response)
pprint.pprint(data)

## Key Takeaways

1. **Schema-first**: Define JSON schema before extraction
2. **Pydantic recommended**: Use Pydantic models for better type safety and IDE support
4. **Validation**: Always validate LLM outputs against schema
5. **Repair**: Common JSON errors can be auto-fixed with `repair_json()`
6. **Token cost**: Validation errors require retry, increasing cost

**Comparison:**
- **Manual JSON Schema**: More control, works with all providers
- **Pydantic + Prompts**: Type-safe, better DX, works with all providers